# Fine-tuning Text-to-SQL no Colab

Use este notebook para executar as CLIs do repositório. A lógica fica em `src/`, `custom_metrics/` e `scripts/`.

O relatório final compara o baseline com os experimentos A–H e recomenda o **Exp F** como compromisso entre Spider e a suíte MMLU 150. O Exp G obteve o maior resultado Spider. Consulte o [README](https://github.com/mh131105/Fine-tuning-de-LLM-para-Text-to-SQL/tree/pipeline) antes de executar.

Para reproduzir o treino reportado, selecione uma GPU A100. Os experimentos finais usam LoRA BF16 sem quantização. Treinar novamente é opcional quando você já dispõe do adapter avaliado; os benchmarks podem carregá-lo de `outputs/<exp>/adapter/`.


## 1. Obter o código

A branch padrão é `pipeline`. A célula clona o repositório quando a pasta ainda não existe e entra nela. Se reutilizar uma sessão, confira a revisão do checkout antes de executar os benchmarks.


In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path('/content/text-to-sql')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--branch', 'pipeline', '--single-branch',
        'https://github.com/mh131105/Fine-tuning-de-LLM-para-Text-to-SQL.git',
        str(repo_dir),
    ], check=True)
%cd /content/text-to-sql
!git branch --show-current
!git rev-parse HEAD


## 2. Instalar dependências

O projeto mantém o PyTorch fornecido pelo runtime Colab. Reinicie a sessão se houver conflito com pacotes de uma instalação anterior.


In [ ]:
!python -m pip install -r requirements.txt


## 3. Autenticação opcional no Hugging Face

Para acesso autenticado, defina `HF_TOKEN` nos segredos do Colab ou no ambiente. A célula não imprime o token nem o adiciona às credenciais do Git. O modelo-base e os datasets são públicos.


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    try:
        hf_token = userdata.get('HF_TOKEN')
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        hf_token = None
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print('Sem token: os próximos comandos usarão acesso público ao Hub.')


## 4. Preparar os dados

O preparo do Spider reaproveita `data/raw/spider/`, quando presente, ou tenta a fonte Hugging Face configurada. Para importar seu próprio diretório ou arquivo ZIP/TAR, use `--source_path` no lugar da aquisição automática.

O preparo do MMLU persiste uma suíte de 150 questões, com 50 de cada uma das três subcategorias em `configs/eval.yaml`. Use a mesma suíte para todos os modelos. Os resultados do relatório não representam o MMLU completo.


In [ ]:
!python -m scripts.prepare_spider --data_dir data/raw/spider --output_dir data/processed/spider
# Alternativa para um Spider já disponível no Drive (monte o Drive antes):
# !python -m scripts.prepare_spider --data_dir data/raw/spider --output_dir data/processed/spider --source_path /content/drive/MyDrive/spider.zip
from pathlib import Path
import shutil
Path('data/processed').mkdir(parents=True, exist_ok=True)
shutil.copy2('reports/evaluation/mmlu_150_suite.json', 'data/processed/mmlu_150_suite.json')
# Para gerar outra suite (preserve a final antes):
# !python -m scripts.prepare_mmlu --config configs/eval.yaml


## 5. Avaliar o baseline

O comando executa Spider e MMLU. Preserve as métricas em `outputs/base/`, pois os benchmarks dos adapters calculam as diferenças em relação a elas.


In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/base


## 6. Treinar os adapters A–H

Execute as células dos experimentos que pretende reproduzir. O **Exp F** usa `q/k/v/o`, learning rate `2e-4` e duas épocas. Todos usam rank 16, alpha 32 e dropout 0,05; as configurações registram as demais diferenças.

Antes de repetir um treino, preserve e mova sua pasta `outputs/<exp>/` anterior para evitar misturar checkpoints e resultados de execuções diferentes. Ao usar um adapter já treinado, pule a célula de treino correspondente. O código salva o adapter em `outputs/<exp>/adapter/`.


In [ ]:
# Experimento A
!python -m scripts.train --config configs/train_lora_exp_a.yaml


In [ ]:
# Experimento B
!python -m scripts.train --config configs/train_lora_exp_b.yaml


In [ ]:
# Experimento C
!python -m scripts.train --config configs/train_lora_exp_c.yaml


In [ ]:
# Experimento D
!python -m scripts.train --config configs/train_lora_exp_d.yaml


In [ ]:
# Experimento E
!python -m scripts.train --config configs/train_lora_exp_e.yaml


In [ ]:
# Experimento F
!python -m scripts.train --config configs/train_lora_exp_f.yaml


In [ ]:
# Experimento G
!python -m scripts.train --config configs/train_lora_exp_g.yaml


In [ ]:
# Experimento H
!python -m scripts.train --config configs/train_lora_exp_h.yaml


## 7. Avaliar os adapters

Execute os benchmarks dos adapters disponíveis. Cada comando avalia Spider dev e a mesma suíte MMLU 150. A geração é greedy, e os parâmetros ficam em `configs/eval.yaml`. Para conferir os resultados reportados, use os 1034 exemplos Spider e as 150 questões MMLU, sem `--limit` e sem `--mock`.

Confira a presença de `outputs/<exp>/adapter/adapter_config.json` e dos pesos antes de avaliar. O carregador usa o modelo-base quando não encontra um adapter, por isso uma pasta vazia não representa um experimento treinado.


In [ ]:
# Experimento A
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_a


In [ ]:
# Experimento B
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_b


In [ ]:
# Experimento C
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_c


In [ ]:
# Experimento D
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_d


In [ ]:
# Experimento E
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_e


In [ ]:
# Experimento F
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_f


In [ ]:
# Experimento G
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_g


In [ ]:
# Experimento H
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_h


## 8. Verificar o código sem baixar o modelo

Os testes usam casos pequenos e mocks. Eles verificam o pipeline, mas não revalidam as acurácias do relatório. Execute demonstrações com `--mock` em diretórios separados das métricas reais e da suíte MMLU preservada.


In [ ]:
!python -m pytest


## 9. Preservar a execução

Antes de encerrar o Colab, copie para armazenamento persistente o adapter escolhido, o tokenizer, a suíte MMLU, as configurações efetivamente utilizadas, os logs, o ambiente, as predições e as métricas. O Git ignora os dados e os arquivos de runtime em `outputs/`; salvar o notebook executado não preserva os pesos.
